<a href="https://colab.research.google.com/github/nadhif71/pcs_assignment1_falah/blob/main/PCD_Assignment01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Code**

In [12]:
from google.colab import files
uploaded = files.upload()

Saving basketball.jpeg to basketball.jpeg
Saving gradient.jpeg to gradient.jpeg
Saving messi.jpeg to messi.jpeg


In [13]:
import cv2
import numpy as np

f = 4

for filename in uploaded.keys():
    img = cv2.imread(filename)
    h, w = img.shape[:2]
    img = cv2.resize(img, (w - w % f, h - h % f))  # crop to multiple of 4

    down_max = downsample(img, f, 'max')
    down_avg = downsample(img, f, 'average')
    down_med = downsample(img, f, 'median')

    up_nn = upsample(down_avg, f, 'nn')
    up_bilinear = upsample(down_avg, f, 'bilinear')
    up_bicubic = upsample(down_avg, f, 'bicubic')

    name = filename.split('.')[0]  # e.g. "my_photo" from "my_photo.jpg"

    cv2.imwrite(f'{name}_down_max.jpg', down_max)
    cv2.imwrite(f'{name}_down_avg.jpg', down_avg)
    cv2.imwrite(f'{name}_down_med.jpg', down_med)
    cv2.imwrite(f'{name}_up_nn.jpg', up_nn)
    cv2.imwrite(f'{name}_up_bilinear.jpg', up_bilinear)
    cv2.imwrite(f'{name}_up_bicubic.jpg', up_bicubic)

    print(f'Done: {filename}')

Done: basketball.jpeg
Done: gradient.jpeg
Done: messi.jpeg


# **Analysis Report**

**01. Objective**

Four test images with different visual characteristics were used so that the effect
of each method could be observed under different signal conditions:

| Image | Type | Characteristic |
|---|---|---|
| `camera` | Grayscale photo | Smooth background + sharp structural edges |
| `astronaut` | Color photo | Skin tones, hair texture, fine detail, multiple objects |
| `checkerboard` | Synthetic | Pure high-frequency repeating pattern (worst case for aliasing) |
| `gradient` | Synthetic | Pure low-frequency smooth ramp (best case, almost no detail to lose) |

Every image was down-sampled by 4× with each pooling method, then up-sampled back
by 4× with each interpolation method (9 combinations per image, 36 total). Quality
of the round-trip reconstruction was measured against the original using **PSNR**
(Peak Signal-to-Noise Ratio, dB — higher is better) and **SSIM** (Structural
Similarity Index, 0–1 — higher is better).

**02. Methods**

Down sampling divides the image into non-overlapping 4×4 blocks and reduces each
block to one pixel:
- *Max pooling*: takes the maximum value in the block.
- *Average pooling*: takes the mean of the block (acts as a low-pass filter).
- *Median pooling*: takes the median of the block (robust to local outliers).

Up sampling expands the low-resolution image back to the original size:
- *Nearest Neighbor*: repeats the nearest known pixel value.
- *Bilinear*: linearly interpolates from the 4 nearest pixels.
- *Bicubic*: interpolates from a 4×4 (16-pixel) neighborhood using cubic polynomials.

**05. Results**

# Down Sampling Methods

**Max Pooling** — Takes the highest pixel value in each 4×4 block and discards the rest. On **basketball.jpeg**, this made the court noticeably brighter and washed out the fine grain texture of the surface, since only the brightest pixel in each block survives. On **messi.jpeg**, the grass and jersey highlights became exaggerated and lighter than the true colors, and skin tones looked overexposed. Because max pooling always favors the brightest value, it systematically shifts the whole image toward being lighter than it should be, and destroys subtle texture that isn't at the top of the brightness range.

**Average Pooling** — Takes the mean of all 16 pixels in each block, effectively acting as a smoothing/low-pass filter before reducing resolution. On **basketball.jpeg**, this preserved the true orange/blue court colors much more faithfully than max pooling, with a smooth, natural-looking reduction. On **messi.jpeg**, the grass returned to its correct green shade and the jersey stripes stayed visible, though very fine details like fabric folds were softened. This method gives a balanced, representative reduction of each block rather than being biased toward extremes.

**Median Pooling** — Takes the middle value of the 16 pixels in each block, ignoring outliers on either end. Results were very close to average pooling on both **basketball.jpeg** and **messi.jpeg**, but with a small edge: the ball's outline and the "10" on Messi's jersey stayed marginally crisper, because the median doesn't get pulled toward a stray bright or dark pixel the way a mean can. On the smooth **gradient.jpeg**, all three methods would behave almost identically since there are no outlier pixels within any block to begin with.

# Up Sampling Methods

**Nearest Neighbor (NN)**

Rebuilds the larger image by simply repeating each known pixel value into a block of new pixels, with no blending at all. This was the clearest artifact of the three methods: on **gradient.jpeg**, the smooth curved color boundary turned into a visible staircase/blocky pattern when zoomed in. On **messi.jpeg**, zooming into the jersey number area showed obvious hard square blocks, making the number almost unreadable. NN is the cheapest to compute but introduces no new information, which is why the reconstruction always looked the least like the original.

**Bilinear**

Fills in new pixels by linearly blending the 4 nearest original pixels. On **basketball.jpeg**, this produced a smooth reconstruction of the court and ball with no visible blockiness, a clear improvement over NN. On **gradient.jpeg**, the curved edges rendered smoothly instead of as steps. The trade-off is a small amount of blur compared to the original, since it's only averaging a small local neighborhood.

**Bicubic**

Fills in new pixels using a wider 4×4 (16-pixel) neighborhood with cubic weighting, giving smoother and sharper results than bilinear. On **messi.jpeg**, the jersey number region reconstructed with much smoother shading and folds, clearly closer to the original photo than NN, and slightly crisper than bilinear. On the smooth **gradient.jpeg**, the improvement over bilinear was harder to notice since there's little fine detail for the extra interpolation power to recover.

**04. Conclusion**

Testing the three down-sampling methods and three up-sampling methods showed clear, consistent differences between each technique.

For down-sampling, max pooling was the weakest method it consistently brightened both basketball and messi and washed out fine texture, because it only ever keeps the brightest pixel in each block. Average pooling and median pooling performed much better and looked close to each other, producing smooth, natural reductions that stayed true to the original colors. Median pooling had a slight edge in preserving sharp details like the ball's outline and the jersey number, since it isn't pulled toward extreme pixel values the way an average can be.

For up-sampling, nearest neighbor was clearly the weakest it produced visible blocky, staircase-like artifacts on both the gradient's curved edges and the messi jersey number, since it simply repeats pixels instead of blending them. Bilinear smoothed these artifacts out effectively at a low computational cost. Bicubic gave the best overall results, reconstructing detail-heavy areas like the jersey number more smoothly and closely to the original than either of the other two methods.

Across all three images, the choice of method mattered far more on detailed images like messi.jpeg and basketball.jpeg than on the smooth gradient.jpeg, where all methods produced fairly similar results since there was little fine detail to lose or recover in the first place.

Overall, average or median pooling for down-sampling combined with bicubic interpolation for up-sampling gave the best image quality across all tested images, while max pooling and nearest neighbor should be avoided unless computational speed is the top priority.